In [62]:
!pip install python-louvain gradio -q

In [63]:
# MainCode.ipynb

import os
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.exceptions import UndefinedMetricWarning
import warnings

from sklearn.metrics import roc_auc_score, average_precision_score
import community.community_louvain as community_louvain

plt.rcParams["figure.figsize"] = (6,4)

# Read clean outputs
df_edges = pd.read_csv("coauthor_edges.csv")
df_authors = pd.read_csv("authors.csv")

df_edges.head(), df_edges["first_year"].min(), df_edges["first_year"].max()

(         author1     author2  first_year  weight
 0       A C Tort  F C Santos        2001       1
 1  A Calogeracos    N Dombey        2000       1
 2  A Calogeracos   P Kennedy        2000       1
 3        A Dadda    M. Billo        1999       1
 4        A Dadda  P. Provero        1999       1,
 1991,
 2003)

In [64]:
def build_full_graph(df_edges, df_authors=None):
    """Build an undirected co-authorship graph.

    - Nodes come from authors.csv (so isolated/single-author nodes are preserved)
    - Edges come from coauthor_edges.csv (author1, author2, first_year, weight)
    """
    G = nx.Graph()

    if df_authors is not None and not df_authors.empty:
        G.add_nodes_from(df_authors["author"].dropna().unique().tolist())

    for a, b, first_year, weight in df_edges[["author1", "author2", "first_year", "weight"]].itertuples(index=False, name=None):
        if pd.isna(a) or pd.isna(b) or a == b:
            continue
        G.add_edge(a, b, first_year=int(first_year), weight=int(weight))

    return G

def basic_stats(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    degs = [d for _, d in G.degree()]
    avg_deg = sum(degs) / n
    clustering = nx.transitivity(G)
    num_cc = nx.number_connected_components(G)

    print("Nodes:", n)
    print("Edges:", m)
    print("Avg degree:", avg_deg)
    print("Connected components:", num_cc)
    print("Global clustering coefficient:", clustering)
    return {
        "nodes": n,
        "edges": m,
        "avg_degree": avg_deg,
        "clustering": clustering,
        "num_cc": num_cc,
    }

G_full = build_full_graph(df_edges, df_authors)
stats_full = basic_stats(G_full)
stats_full

Nodes: 13003
Edges: 23847
Avg degree: 3.6679227870491427
Connected components: 2158
Global clustering coefficient: 0.2396160788529012


{'nodes': 13003,
 'edges': 23847,
 'avg_degree': 3.6679227870491427,
 'clustering': 0.2396160788529012,
 'num_cc': 2158}

In [65]:
SPLIT_YEAR = 1998  # Có thể chỉnh (vd 1999, 2000...)

def temporal_split(df_edges, df_authors, split_year):
    df_train = df_edges[df_edges["first_year"] <= split_year].copy()
    df_test  = df_edges[df_edges["first_year"] >  split_year].copy()

    G_train = nx.Graph()
    if df_authors is not None and not df_authors.empty:
        G_train.add_nodes_from(df_authors["author"].dropna().unique().tolist())

    for a, b, first_year, weight in df_train[["author1","author2","first_year","weight"]].itertuples(index=False, name=None):
        if pd.isna(a) or pd.isna(b) or a == b:
            continue
        G_train.add_edge(a, b, first_year=int(first_year), weight=int(weight))

    train_nodes = set(G_train.nodes())

    E_test_pos = []
    for u, v in df_test[["author1","author2"]].itertuples(index=False, name=None):
        if u in train_nodes and v in train_nodes and not G_train.has_edge(u, v):
            E_test_pos.append((u, v))

    return G_train, E_test_pos, df_train, df_test

G_train, E_test_pos, df_train, df_test = temporal_split(df_edges, df_authors, SPLIT_YEAR)

print("Train edges:", len(df_train), "Test pos edges:", len(E_test_pos))
basic_stats(G_train);

Train edges: 13162 Test pos edges: 10685
Nodes: 13003
Edges: 13162
Avg degree: 2.024455894793509
Connected components: 6118
Global clustering coefficient: 0.3125395569620253


In [66]:
import random

def sample_negative_edges(G, num_samples, node_pool=None, max_tries=100000):
    nodes = list(node_pool) if node_pool is not None else list(G.nodes())
    neg = set()
    tries = 0
    while len(neg) < num_samples and tries < max_tries:
        u, v = random.sample(nodes, 2)
        if not G.has_edge(u, v):
            neg.add(tuple(sorted((u, v))))
        tries += 1
    return list(neg)

def evaluate_lp(G_train, E_pos, score_func, num_neg=None, seed=42, node_pool=None):
    random.seed(seed)
    if num_neg is None:
        num_neg = len(E_pos)

    E_neg = sample_negative_edges(G_train, num_neg, node_pool=node_pool)

    y_true = [1]*len(E_pos) + [0]*len(E_neg)
    scores = []

    for (u, v) in E_pos:
        scores.append(score_func(G_train, u, v))
    for (u, v) in E_neg:
        scores.append(score_func(G_train, u, v))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        auc = roc_auc_score(y_true, scores)
        ap  = average_precision_score(y_true, scores)

    return auc, ap

In [67]:
def common_neighbors(G, u, v):
    return len(list(nx.common_neighbors(G, u, v)))

def adamic_adar(G, u, v):
    cn = nx.common_neighbors(G, u, v)
    s = 0.0
    for z in cn:
        deg = G.degree(z)
        if deg > 1:
            s += 1.0 / np.log(deg)
    return s

def resource_allocation(G, u, v):
    cn = nx.common_neighbors(G, u, v)
    s = 0.0
    for z in cn:
        deg = G.degree(z)
        if deg > 0:
            s += 1.0 / deg
    return s

def preferential_attachment(G, u, v):
    return G.degree(u) * G.degree(v)

In [68]:
# Louvain trên G_train
partition = community_louvain.best_partition(G_train, random_state=42)
modularity = community_louvain.modularity(partition, G_train)
print("Louvain modularity:", modularity)

def make_same_comm_scorer(part):
    def same_comm(G, u, v):
        if u not in part or v not in part:
            return 0.0
        return 1.0 if part[u] == part[v] else 0.0
    return same_comm

same_comm = make_same_comm_scorer(partition)

Louvain modularity: 0.9214819075606969


In [69]:
def make_hybrid(base_func, partition, alpha=0.0):
    def hybrid(G, u, v):
        base = base_func(G, u, v)
        same = 1.0 if partition.get(u) == partition.get(v) else 0.0
        return base + (alpha * same)
    return hybrid


# Create hybrid variants (OFFLINE evaluation)
HYB_CN = make_hybrid(common_neighbors, partition, alpha=2.0)
HYB_AA = make_hybrid(adamic_adar, partition, alpha=2.0)
HYB_RA = make_hybrid(resource_allocation, partition, alpha=2.0)
HYB_PA = make_hybrid(preferential_attachment, partition, alpha=2.0)

In [70]:
methods = [
    ("CN", common_neighbors),
    ("AA", adamic_adar),
    ("RA", resource_allocation),
    ("PA", preferential_attachment),
    ("Louvain", same_comm),
]

results = []
node_pool_train_active = set(df_train["author1"]).union(set(df_train["author2"]))
for name, func in methods:
    auc, ap = evaluate_lp(G_train, E_test_pos, func, node_pool=node_pool_train_active)
    print(f"{name}: AUC={auc:.4f}, AP={ap:.4f}")
    results.append((name, auc, ap))

df_results = pd.DataFrame(results, columns=["Method", "AUC", "AP"])
df_results

CN: AUC=0.5210, AP=0.5206
AA: AUC=0.5210, AP=0.5209
RA: AUC=0.5210, AP=0.5207
PA: AUC=0.1946, AP=0.5008
Louvain: AUC=0.5257, AP=0.5214


,Method,AUC,AP
0,CN,0.521019,0.520592
1,AA,0.521020,0.520892
2,RA,0.521018,0.520699
3,PA,0.194645,0.500764
4,Louvain,0.525690,0.521414


In [71]:
hyb_methods = [
    ("HYB_CN", HYB_CN),
    ("HYB_AA", HYB_AA),
    ("HYB_RA", HYB_RA),
    ("HYB_PA", HYB_PA),
]

hyb_results = []
for name, func in hyb_methods:
    auc, ap = evaluate_lp(
        G_train,
        E_test_pos,
        func,
        node_pool=node_pool_train_active
    )
    print(f"{name}: AUC={auc:.4f}, AP={ap:.4f}")
    hyb_results.append((name, auc, ap))

df_hyb = pd.DataFrame(hyb_results, columns=["Method", "AUC", "AP"])
df_hyb

HYB_CN: AUC=0.5302, AP=0.5296
HYB_AA: AUC=0.5301, AP=0.5296
HYB_RA: AUC=0.5301, AP=0.5295
HYB_PA: AUC=0.1966, AP=0.5023


,Method,AUC,AP
0,HYB_CN,0.530150,0.529575
1,HYB_AA,0.530144,0.529620
2,HYB_RA,0.530143,0.529480
3,HYB_PA,0.196602,0.502346


In [72]:
def two_hop_candidates(G, u):
    nbrs = set(G.neighbors(u))
    cand = set()
    for w in nbrs:
        cand |= set(G.neighbors(w))
    cand.discard(u)
    cand -= nbrs
    return list(cand)

def recommend(G, u, score_func, min_score=0.0):
    if u not in G:
        raise ValueError(f"Author '{u}' not in graph.")

    cand = two_hop_candidates(G, u)
    scored = []
    for v in cand:
        s = score_func(G, u, v)
        if s > min_score:
            scored.append((v, s))
    return scored

def explain_recommendations(G, u, score_func, k=10):
    recs = recommend(G, u, score_func)

    # Tính đường đi ngắn nhất ≤4 từ u tới mọi node một lần (nhanh hơn rất nhiều)
    try:
        paths_from_u = nx.single_source_shortest_path(G, source=u, cutoff=4)
    except Exception:
        paths_from_u = {}

    results = []
    for v, sc in recs:
        # Common neighbors
        cn = sorted(list(nx.common_neighbors(G, u, v)))
        cn_names = ", ".join(cn[:30])

        # Shortest path string if exists (≤4)
        if v in paths_from_u:
            sp_path = " → ".join(paths_from_u[v])
        else:
            sp_path = None

        # QUAN TRỌNG: append vào results
        results.append({
            "candidate": v,
            "score": float(sc),
            "common_neighbor_names": cn_names,
            "shortest_path": sp_path
        })

    df = pd.DataFrame(results)
    if df.empty:
        return pd.DataFrame(columns=["candidate","score","common_neighbor_names","shortest_path"])

    # Tie-break: score ↓, cn non-empty first, path exists first, then candidate
    df["has_cn"] = df["common_neighbor_names"].apply(lambda x: 0 if (pd.isna(x) or str(x).strip() == "") else 1)
    df["has_path"] = df["shortest_path"].apply(lambda x: 0 if (pd.isna(x) or str(x).strip() == "") else 1)

    df = df.sort_values(
        by=["score", "has_cn", "has_path", "candidate"],
        ascending=[False, False, False, True],
        kind="mergesort"
    ).head(int(k)).reset_index(drop=True)

    return df.drop(columns=["has_cn", "has_path"])

In [73]:
EXPECTED_COLS = [
    "method",
    "rank",
    "candidate",
    "score",
    "common_neighbor_names",
    "shortest_path",
]

In [74]:
import pandas as pd
import gradio as gr

# 1) Danh sách methods
method_dict = {
    "CN": common_neighbors,
    "AA": adamic_adar,
    "RA": resource_allocation,
    "PA": preferential_attachment,
    "HYB_AA": HYB_AA,   # <-- hybrid representative
}

def run_all_methods(author_name: str, k: int):
    author_name = (author_name or "").strip()
    if author_name not in G_train:
        df_err = pd.DataFrame([{
            "method": "ERROR",
            "rank": "",
            "candidate": "",
            "score": "",
            "common_neighbor_names": "",
            "shortest_path": f"Author '{author_name}' not found in graph."
        }])
        return df_err.reindex(columns=EXPECTED_COLS)

    rows = []
    for mname, func in method_dict.items():
        try:
            df = explain_recommendations(G_train, author_name, func, k=int(k)).copy()
        except Exception as e:
            # nếu method nào lỗi, vẫn hiển thị lỗi ngay trong bảng
            df = pd.DataFrame([{
                "candidate": f"[{mname}] ERROR",
                "score": "",
                "common_neighbor_names": "",
                "shortest_path": str(e),
            }])

        if df is None or len(df) == 0:
            continue

        df.insert(0, "method", mname)
        df.insert(1, "rank", range(1, len(df) + 1))

        # đảm bảo đủ cột đúng schema mới
        df = df.reindex(columns=EXPECTED_COLS)

        rows.append(df)

    out = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=EXPECTED_COLS)
    if len(out) > 0:
        out = out.sort_values(["method", "rank"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    return out

In [75]:
def agreement_table(compare_df: pd.DataFrame):
    if compare_df is None or len(compare_df) == 0:
        return pd.DataFrame(columns=["candidate","n_methods","methods"])

    compare_df = compare_df[compare_df["method"] != "ERROR"].copy()
    compare_df = compare_df[compare_df["candidate"].notna()].copy()
    compare_df = compare_df[compare_df["candidate"].astype(str).str.strip() != ""].copy()
    compare_df = compare_df[~compare_df["candidate"].astype(str).str.contains("ERROR", na=False)].copy()

    if len(compare_df) == 0:
        return pd.DataFrame(columns=["candidate","n_methods","methods"])

    g = (compare_df.groupby("candidate")["method"]
         .agg(n_methods="nunique", methods=lambda s: ", ".join(sorted(set(s)))))
    return g.reset_index().sort_values(["n_methods","candidate"], ascending=[False, True])

In [76]:
CSS = """
#title {font-size: 28px; font-weight: 800; margin-bottom: 4px;}
#subtitle {color: #555; margin-bottom: 16px;}
.card {border: 1px solid #eee; border-radius: 14px; padding: 14px;}
"""

with gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:
    gr.Markdown("Author Collaborator Finder (Hep-Th Coauthor Network)", elem_id="title")
    gr.Markdown("Compare proximity-based heuristics and a community-aware hybrid framework",elem_id="subtitle")


    with gr.Row():
        with gr.Column(scale=2):
            author_in = gr.Textbox(label="Author name (exact)", placeholder="e.g., 'E. Witten'")
        with gr.Column(scale=1):
            k_in = gr.Slider(1, 20, value=5, step=1, label="Top-K")

    run_btn = gr.Button("Run comparison", variant="primary")

    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### Comparison (all methods)", elem_classes=["card"])
            compare_out = gr.Dataframe(
                headers=EXPECTED_COLS,
                label="",
                wrap=True,
                interactive=False,
            )

        with gr.Column(scale=1):
            gr.Markdown("### Agreement (consensus)", elem_classes=["card"])
            agree_out = gr.Dataframe(
                headers=["candidate","n_methods","methods"],
                label="",
                wrap=True,
                interactive=False,
            )

    def run_and_agree(author_name, k):
        comp = run_all_methods(author_name, k)
        agr = agreement_table(comp)
        return comp, agr


    run_btn.click(run_and_agree, inputs=[author_in, k_in], outputs=[compare_out, agree_out])

demo.launch()

/tmp/ipython-input-1808543418.py:7: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:
/tmp/ipython-input-1808543418.py:7: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf17b0f51b17392d96.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
